# Нормализация пустых строк

Тематический этап подготовки Европы 2019–2023. Выполняйте ячейки сверху вниз в окружении проекта. Каждый ноутбук соответствует одному шагу; диагностика и применение согласованного изменения разделены. Входы, выходы и порядок описаны в [карте подготовки](README.md). Сохраненные результаты перенесены из выполненного общего ноутбука; сам перенос данных не преобразует.

## Шаг 2 – Согласованная нормализация пустых строк

Согласовано: только в `registration`, `typecode` и `icaoaircrafttype` заменяем точную пустую строку на физический null. Сохраняем три булевых поля `<имя>_was_empty_string`: True означает исходную пустую строку, False – любое другое исходное значение, включая исходный null. Вместе с неизмененным источником это позволяет восстановить исходное представление.

Записи и остальные значения сохраняются. Это промежуточная рабочая копия после одного правила, а не окончательная аналитическая база. Заполнение пропусков и другие правила еще не согласованы.

### Подготовка входа

Повторяем проверку источника и подсчет пропусков из 03_01, чтобы выполнять изменение в отдельном ядре.

In [1]:
# Подключаем библиотеки для чтения Parquet, таблиц и проверки целостности
from pathlib import Path
import hashlib
import json
from collections import Counter
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

# Находим корень проекта независимо от папки запуска ноутбука
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'europe/data/source/osn23_landings_merged.parquet.gzip').exists())
SOURCE = ROOT / 'europe/data/source/osn23_landings_merged.parquet.gzip'

# Считаем SHA256 блоками: весь исходный файл не нужно загружать в память
def source_hash():
    h = hashlib.sha256()
    with SOURCE.open('rb') as stream:
        for block in iter(lambda: stream.read(4 * 1024**2), b''):
            h.update(block)
    return h.hexdigest()

# Сверяем источник и число строк с уже выполненным аудитом
before = source_hash()
audit = json.loads((ROOT / 'europe/reports/profile_summary.json').read_text(encoding='utf8'))
assert before == audit['source_sha256'], 'Источник отличается от файла аудита 01.'
parquet = pq.ParquetFile(SOURCE)
assert parquet.metadata.num_rows == audit['rows']
print('Записей:', f'{parquet.metadata.num_rows:,}'.replace(',', ' '))
print('SHA256 совпадает с аудитом 01.')

Записей: 7 984 254
SHA256 совпадает с аудитом 01.


In [2]:
# Выбираем строковые поля верхнего уровня; вложенные списки здесь не проверяем
fields = [f.name for f in parquet.schema_arrow
          if pa.types.is_string(f.type) or pa.types.is_large_string(f.type)]
counts = {field: Counter() for field in fields}
examples = []
shown = Counter()
# Читаем по 100 000 строк и сохраняем глобальную позицию примеров
offset = 0
for batch in parquet.iter_batches(batch_size=100_000, columns=fields):
    frame = batch.to_pandas(ignore_metadata=True)
    for field in fields:
        values = frame[field]
        # Различаем null, точную пустую строку и строку из пробелов
        masks = {
            'null': values.isna(),
            'empty_string': values.eq('').fillna(False),
            'whitespace_only': (values.notna() & values.ne('')
                                & values.str.strip().eq('')).fillna(False),
        }
        for kind, mask in masks.items():
            counts[field][kind] += int(mask.sum())
            # Ограничиваем примеры тремя строками на поле и вид пропуска
            remaining = 3 - shown[(field, kind)]
            if remaining > 0:
                for position in frame.index[mask][:remaining]:
                    examples.append({'field': field, 'kind': kind,
                                     'source_row_position': offset + int(position),
                                     'raw_repr': repr(values.loc[position])})
                    shown[(field, kind)] += 1
    offset += len(frame)

# Проверяем полный охват файла и рассчитываем доли от всех строк
assert offset == audit['rows']
missingness = pd.DataFrame([{'field': field, **dict(counts[field])} for field in fields])
for kind in ['null', 'empty_string', 'whitespace_only']:
    missingness[kind + '_percent'] = 100 * missingness[kind] / offset
assert source_hash() == before, 'Контрольная сумма источника изменилась.'
display(missingness)
print('Исходный файл не изменился. Преобразования не выполнялись.')

,field,null,empty_string,whitespace_only,null_percent,empty_string_percent,whitespace_only_percent
0,flight_id,0,0,0,0.000000,0.000000,0.0
1,callsign,0,0,0,0.000000,0.000000,0.0
2,icao24,0,0,0,0.000000,0.000000,0.0
3,registration,859077,242227,0,10.759640,3.033809,0.0
4,typecode,859077,475015,0,10.759640,5.949397,0.0
5,icaoaircrafttype,859077,543159,0,10.759640,6.802877,0.0
6,airport,0,0,0,0.000000,0.000000,0.0
7,country,0,0,0,0.000000,0.000000,0.0
8,market_segment,1515276,0,0,18.978304,0.000000,0.0
9,AC_CLASS,1652382,0,0,20.695509,0.000000,0.0


Исходный файл не изменился. Преобразования не выполнялись.


In [4]:
# Подключаем операции Arrow для преобразования без потери типов
import pyarrow.compute as pc

# Нормализуем только три согласованных поля; создаем отдельную рабочую копию
approved_fields = ['registration', 'typecode', 'icaoaircrafttype']
WORK = ROOT / 'europe/data/working'
WORK.mkdir(parents=True, exist_ok=True)
DESTINATION = WORK / '01_empty_strings_normalized.parquet'
TEMPORARY = WORK / '01_empty_strings_normalized.parquet.part'
output_schema = parquet.schema_arrow
for field in approved_fields:
    assert field + '_was_empty_string' not in output_schema.names
    output_schema = output_schema.append(pa.field(field + '_was_empty_string', pa.bool_()))

replaced = Counter()
written = 0
# Сначала пишем временный файл; для каждой замены сохраняем обратимый флаг
with pq.ParquetWriter(TEMPORARY, output_schema, compression='gzip', compression_level=1) as writer:
    for batch in parquet.iter_batches(batch_size=100_000):
        table = pa.Table.from_batches([batch])
        for field in approved_fields:
            original = table[field]
            mask = pc.fill_null(pc.equal(original, ''), False)
            replaced[field] += pc.sum(pc.cast(mask, pa.int64())).as_py()
            normalized = pc.if_else(mask, pa.scalar(None, type=original.type), original)
            table = table.set_column(table.schema.get_field_index(field),
                                     table.schema.field(field), normalized)
            table = table.append_column(field + '_was_empty_string', mask)
        writer.write_table(table)
        written += len(table)

# Проверяем обратимость каждого изменения и неизменность остальных полей
candidate = pq.ParquetFile(TEMPORARY)
assert candidate.metadata.num_rows == written == audit['rows']
checked = 0
after_nulls = Counter()
for source_batch, result_batch in zip(parquet.iter_batches(batch_size=100_000),
                                      candidate.iter_batches(batch_size=100_000), strict=True):
    original = pa.Table.from_batches([source_batch])
    result = pa.Table.from_batches([result_batch])
    for field in approved_fields:
        flag = result[field + '_was_empty_string']
        expected_flag = pc.fill_null(pc.equal(original[field], ''), False)
        assert flag.equals(expected_flag)
        assert pc.sum(pc.cast(pc.fill_null(pc.equal(result[field], ''), False), pa.int64())).as_py() == 0
        after_nulls[field] += result[field].null_count
        restored = pc.if_else(flag, pa.scalar('', type=original[field].type), result[field])
        result = result.set_column(result.schema.get_field_index(field), original.schema.field(field), restored)
    assert result.select(original.column_names).equals(original)
    checked += len(original)
assert checked == written
assert source_hash() == before
for field in approved_fields:
    assert replaced[field] == counts[field]['empty_string']
    assert after_nulls[field] == counts[field]['null'] + replaced[field]

# После всех проверок публикуем рабочую копию заменой временного файла
candidate.close()
TEMPORARY.replace(DESTINATION)
change_log = pd.DataFrame([{'field': field, 'empty_strings_replaced': replaced[field],
                            'null_before': counts[field]['null'], 'null_after': after_nulls[field]}
                           for field in approved_fields])
display(change_log)
# Сохраняем счетчики и контрольные суммы в журнал воспроизводимости
REPORT = ROOT / 'europe/reports/preparation'
REPORT.mkdir(parents=True, exist_ok=True)
change_log.to_csv(REPORT / '01_empty_strings_normalized.csv', index=False)
(REPORT / '01_empty_strings_normalized.json').write_text(json.dumps({
    'rule': 'exact empty string to null in three approved fields',
    'approved_fields': approved_fields,
    'source_sha256': before, 'source_unchanged': True,
    'output': str(DESTINATION.relative_to(ROOT)),
    'output_sha256': hashlib.file_digest(DESTINATION.open('rb'), 'sha256').hexdigest(),
    'rows_before': written, 'rows_after': written, 'rows_deleted': 0,
    'reversible_comparison_passed': True,
    'changes': change_log.to_dict(orient='records'),
}, ensure_ascii=False, indent=2) + '\n', encoding='utf8')
print('Сохранена рабочая копия:', DESTINATION)
print('Все записи сохранены. Проверена обратимость и неизменность остальных полей.')

,field,empty_strings_replaced,null_before,null_after
0,registration,242227,859077,1101304
1,typecode,475015,859077,1334092
2,icaoaircrafttype,543159,859077,1402236


Сохранена рабочая копия: D:\AI-ML Engineering\Projects\5 - Approach Risk ML\Approach Risk\europe\data\working\01_empty_strings_normalized.parquet
Все записи сохранены. Проверена обратимость и неизменность остальных полей.
